# GUSALUD en Google Colab

Este cuaderno inicia la API y el panel principal usando los archivos del proyecto. Cuando se solicite, carga `GUSALUD_vacop_ejecutable.zip` desde tu computador. El paquete incluye una base pequeña de ejemplo; puedes subir tu CSV/Excel en la celda opcional para que los indicadores se calculen con esos datos.

LM Studio no es necesario para iniciar. Sin LM Studio verás igualmente los cálculos de Python; solo faltará la narración generada por el modelo.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os, json, subprocess, sys, time

print("Sube GUSALUD_vacop_ejecutable.zip")
subidos = files.upload()
zip_subido = next((Path(nombre) for nombre in subidos if nombre.lower().endswith(".zip")), None)
if zip_subido is None:
    raise ValueError("No se cargó un archivo ZIP del proyecto.")

PROYECTO = Path("/content/gusalud")
PROYECTO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_subido) as paquete:
    miembros = paquete.namelist()
    if any(Path(n).is_absolute() or ".." in Path(n).parts for n in miembros):
        raise ValueError("El ZIP contiene rutas no permitidas.")
    paquete.extractall(PROYECTO)

requeridos = ["vacunacion_guajira.py", "index.html", "styles.css", "requirements.txt", "vacunacion_datos_2000.csv"]
no_estan = [n for n in requeridos if not (PROYECTO / n).is_file()]
if no_estan:
    raise FileNotFoundError("El ZIP no contiene: " + ", ".join(no_estan))
# El API usa este archivo de ejemplo al iniciar; se puede reemplazar en la celda opcional.
(PROYECTO / ".vacunacion_config.json").write_text(json.dumps({"archivo": str(PROYECTO / "vacunacion_datos_2000.csv")}), encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROYECTO / "requirements.txt")])
os.chdir(PROYECTO)
print("GUSALUD preparado en", PROYECTO)


## (Opcional) Usar tus datos
Ejecuta esta celda si deseas que la API lea otro archivo CSV o Excel. El archivo debe incluir los encabezados usados por GUSALUD. La selección se guarda como fuente activa del análisis.


In [ ]:
from google.colab import files
from pathlib import Path
import json, shutil

subidos_datos = files.upload()
archivo_datos = next((Path(n) for n in subidos_datos if n.lower().endswith((".csv", ".xlsx", ".xlsm"))), None)
if archivo_datos is None:
    raise ValueError("Carga un archivo CSV o Excel.")
destino_datos = PROYECTO / archivo_datos.name
if archivo_datos.resolve() != destino_datos.resolve():
    shutil.copy2(archivo_datos, destino_datos)
(PROYECTO / ".vacunacion_config.json").write_text(json.dumps({"archivo": str(destino_datos)}), encoding="utf-8")
print("Fuente de datos configurada:", destino_datos.name)


In [ ]:
import subprocess, sys, time, requests
from IPython.display import HTML, display

# Evita lanzar dos copias si se vuelve a ejecutar esta celda.
try:
    proceso_api
    if proceso_api.poll() is not None:
        raise NameError
except NameError:
    proceso_api = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "vacunacion_guajira:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=PROYECTO, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
    )

for intento in range(30):
    try:
        respuesta = requests.get("http://127.0.0.1:8000/salud", timeout=2)
        if respuesta.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("La API no inició; reinicia el entorno y ejecuta las celdas en orden.")

from google.colab import output
enlace = output.eval_js("google.colab.kernel.proxyPort(8000)")
print("API activa:", respuesta.json())
display(HTML(f'<p><a href="{enlace}" target="_blank" style="font-size:18px;font-weight:700">Abrir panel GUSALUD</a></p>'))


### Uso
El panel vuelve a consultar el archivo registrado al pulsar **Actualizar datos**. Si agregas o modificas registros desde el backend, guarda primero los cambios y luego actualiza el panel. Los archivos que subes viven en el entorno temporal de Colab; descarga cualquier resultado que quieras conservar antes de cerrar la sesión.
